# UCS420 Lab Assignment 9 — NLP Text Preprocessing
Enter your roll number and name, then run all cells.

In [ ]:
ROLL_NO = 1024170198         # <-- your full roll number
NAME = "Zaid Yusuf"            # <-- your name

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
assert ROLL_NO > 0 and NAME, "Enter your roll number and name first"

### Dataset generator — do not modify

In [ ]:
import random, hashlib
import pandas as pd

def generate_dataset(roll_no):
    """Return (DataFrame of tickets, focus ticket IDs, fingerprint) for a roll number. Do not modify."""
    rng = random.Random(int(roll_no))

    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }

    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)

    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint

### Preprocessing pipeline — do not modify

In [ ]:
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def is_punct(token):
    """A token is punctuation if EVERY character in it is in string.punctuation (e.g. '.', '...', '``', "''")."""
    return all(ch in string.punctuation for ch in token)

def process_ticket(text):
    """The official pipeline. Returns sentences, lowercase tokens, punctuation, stopwords and final tokens."""
    sentences = sent_tokenize(text)
    tokens = [t.lower() for s in sentences for t in word_tokenize(s)]
    punct = [t for t in tokens if is_punct(t)]
    stops = [t for t in tokens if t in STOP]
    final = [t for t in tokens if not is_punct(t) and t not in STOP]
    return {"sentences": sentences, "tokens": tokens, "punct": punct, "stops": stops, "final": final}

In [ ]:
df, FOCUS, FINGERPRINT = generate_dataset(ROLL_NO)
print(f"Roll number: {ROLL_NO}   Name: {NAME}   Fingerprint: {FINGERPRINT}   Tickets: {len(df)}\n")
print(df.to_string(index=False))
df.to_csv(f"{ROLL_NO}_generated_dataset.csv", index=False)

---
## Q1. Preprocessing pipeline
Run process_ticket() on every ticket and show one table with, per ticket: sentences, tokens, punctuation tokens, stopwords, unique tokens and final tokens, plus a TOTAL row.

In [ ]:
processed = [process_ticket(t) for t in df["Ticket_Text"]]
rows = []

for i, result in enumerate(processed):
    rows.append({
        "Ticket": df.loc[i, "Ticket_ID"],
        "Sentences": len(result["sentences"]),
        "Tokens": len(result["tokens"]),
        "Punctuation Tokens": len(result["punct"]),
        "Stopwords": len(result["stops"]),
        "Unique Tokens": len(set(result["tokens"])),
        "Final Tokens": len(result["final"])
    })

result_df = pd.DataFrame(rows)
count_cols = ["Sentences", "Tokens", "Punctuation Tokens", "Stopwords", "Final Tokens"]
total_row = {"Ticket": "TOTAL", "Unique Tokens": None}
for col in count_cols:
    total_row[col] = result_df[col].sum()
result_df = pd.concat([result_df, pd.DataFrame([total_row])], ignore_index=True)
display(result_df)

sample_id = FOCUS[0]
index = df.index[df["Ticket_ID"] == sample_id][0]
sample = processed[index]
print("Selected ticket:", sample_id)
print("Original text:", df.loc[index, "Ticket_Text"])
print("Sentences:", sample["sentences"])
print("Tokens:", sample["tokens"])
print("Punctuation:", sample["punct"])
print("Stopwords:", sample["stops"])
print("Final tokens:", sample["final"])

odd_tokens = sorted({w for p in processed for w in p["final"]
                     if w in ["n't", "'s", "ca", "wo", "did", "``", "''"]})
print("Some unusual final tokens:", odd_tokens)
print("These are mainly produced when word_tokenize separates contractions.")

**Observe:** The tokenizer can break contractions into parts such as `ca` and `n't`. Some parts do not work as meaningful words by themselves.

---
## Q2. Regular expressions
On your raw tickets, use re.findall to extract: words longer than 5 letters; all numbers (keep 95.5 and 2.0.1 whole); capitalized words; words starting with a vowel. Then use re.sub to replace emails with <EMAIL>, URLs with <URL> and phone numbers with <PHONE>. Your patterns must also work on TEST_LINES (given in the notebook).

In [ ]:
import re

TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]

email_pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
url_pattern = r"(?:https?://|www\.)[^\s,;]+"
phone_pattern = r"(?<!\w)(?:\+91[ -]?)?(?:\d{5}[ -]?\d{5}|\d{4}-\d{3}-\d{4})(?!\w)"
number_pattern = r"(?<![A-Za-z0-9])\d+(?:\.\d+)*(?![A-Za-z0-9])"

all_lines = list(df["Ticket_Text"]) + TEST_LINES
for i, line in enumerate(all_lines, 1):
    print("\nTEXT", i, line)
    print("Long words:", re.findall(r"\b[A-Za-z]{6,}\b", line))
    print("Numbers:", re.findall(number_pattern, line))
    print("Capitalized:", re.findall(r"\b[A-Z][a-z]+\b", line))
    print("Vowel words:", re.findall(r"\b[AEIOUaeiou][A-Za-z]*\b", line))

    changed = re.sub(email_pattern, "<EMAIL>", line)
    changed = re.sub(url_pattern, "<URL>", changed)
    changed = re.sub(phone_pattern, "<PHONE>", changed)
    print("Replaced:", changed)

print("\nTEST LINES CHECK")
for line in TEST_LINES:
    print("Original:", line)
    s = re.sub(email_pattern, "<EMAIL>", line)
    s = re.sub(url_pattern, "<URL>", s)
    s = re.sub(phone_pattern, "<PHONE>", s)
    print("After:", s)

capitalized = re.findall(r"\b[A-Z][a-z]+\b", " ".join(df["Ticket_Text"]))
start_caps = []
for t in df["Ticket_Text"]:
    for sentence in sent_tokenize(t):
        first = re.search(r"\b[A-Z][a-z]+\b", sentence)
        if first and first.start() == len(sentence) - len(sentence.lstrip()):
            start_caps.append(first.group())
print("Capitalized words at sentence start:", len(start_caps))

**Observe:** Some capitalized words are only capitalized because they come at the start of a sentence. The code above counts these separately.

---
## Q3. Custom tokenizer
Write my_tokenize(text) with one RegexpTokenizer pattern that keeps contractions (isn't), hyphenated words (Wi-Fi, state-of-the-art), decimals and versions (95.5, 2.0.1), emails, URLs and phone numbers as single tokens. Run it on TEST_LINES and on your tickets, and compare with word_tokenize.

In [ ]:
from nltk.tokenize import RegexpTokenizer, word_tokenize

# The order matters: match URLs and emails before ordinary words
pattern = (r"(?:https?://|www\.)[^\s,;!?]+(?<![.])"
           r"|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
           r"|(?<!\w)(?:\+91[ -]?)?(?:\d{5}[ -]?\d{5}|\d{4}-\d{3}-\d{4})(?!\w)"
           r"|\d+(?:\.\d+)+"
           r"|[A-Za-z]+(?:[-'][A-Za-z]+)*"
           r"|\d+"
           r"|[^\w\s]")

tokenizer = RegexpTokenizer(pattern)
def my_tokenize(text):
    return tokenizer.tokenize(text)

print("TEST LINES")
for line in TEST_LINES:
    print("\nOriginal:", line)
    print("My tokenizer:", my_tokenize(line))
    print("NLTK tokenizer:", word_tokenize(line))

print("\nTICKET COMPARISON")
different = 0
for i, t in enumerate(df["Ticket_Text"]):
    my_words = my_tokenize(t)
    nltk_words = word_tokenize(t)
    if my_words != nltk_words:
        different += 1
    print(df.loc[i, "Ticket_ID"], "custom:", my_words)
    print("  nltk:", nltk_words)

print("Tickets with different tokenization:", different)
print("Difference 1: contractions like isn't stay together in my tokenizer")
print("Difference 2: hyphenated words like Wi-Fi stay together")
print("Difference 3: versions, email addresses and URLs stay together")

**Observe:** The count of different ticket tokenizations is printed by the code. The custom tokenizer keeps hyphenated words and contractions together while `word_tokenize()` often splits them.

---
## Q4. Stemming and lemmatization
On your unique final tokens, compare PorterStemmer, LancasterStemmer, a RegexpStemmer with a suffix rule you choose after looking at your own words, and WordNetLemmatizer with POS tags from nltk.pos_tag. Show a table of the words where the outputs differ.

In [ ]:
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet

unique_words = sorted(set(w for p in processed for w in p["final"] if w.isalpha()))
porter = PorterStemmer()
lancaster = LancasterStemmer()
lemma = WordNetLemmatizer()

# 'ing' and 'ed' are common endings of actions in IT support tickets
regex_stemmer = RegexpStemmer(r"(ing|ed)$", min=5)

def get_wordnet_pos(tag):
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("N"):
        return wordnet.NOUN
    if tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

all_tags = pos_tag(unique_words)
rows = []
for word, tag in all_tags:
    p = porter.stem(word)
    l = lancaster.stem(word)
    r = regex_stemmer.stem(word)
    w = lemma.lemmatize(word, get_wordnet_pos(tag))
    if len({p, l, r, w}) > 1:
        rows.append({"Word": word, "Porter": p, "Lancaster": l,
                     "Regexp": r, "WordNet": w, "POS": tag})

compare_df = pd.DataFrame(rows, columns=["Word", "Porter", "Lancaster", "Regexp", "WordNet", "POS"])
display(compare_df)

suffix_examples = [w for w in unique_words if w.endswith(("ing", "ed"))]
print("Words supporting my regexp rule:", suffix_examples[:10])
print("The regexp rule removes ing and ed from words in the tickets")

if len(suffix_examples) >= 2:
    print("Examples from dataset:", suffix_examples[0], suffix_examples[1])
else:
    print("Fewer than two examples found, check the generated ticket words")

print("Possible over-stemming examples:")
print(compare_df[compare_df["Lancaster"] != compare_df["WordNet"]].head(5).to_string(index=False))
print("Possible under-stemming examples:")
print(compare_df[compare_df["Regexp"] == compare_df["Word"]].head(5).to_string(index=False))

**Observe:** Lancaster may shorten some words too much, which is over-stemming. The simple Regexp stemmer only removes `ing` or `ed` so it may leave other forms unchanged, which is under-stemming. Specific words are printed from my generated tickets above.

---
## Q5. Corpus statistics
Compute tokens, unique tokens and TTR for S1 (all tokens), S2 (no punctuation), S3 (final tokens) and S4 (S3 after Porter). Print the top 10 words of S1 and S3, plot log(rank) vs log(frequency) for S3, and print the top 5 bigrams of S1 and S3.

In [ ]:
from collections import Counter
from nltk.util import bigrams
import numpy as np
import matplotlib.pyplot as plt

S1 = [w for p in processed for w in p["tokens"]]
S2 = [w for w in S1 if not is_punct(w)]
S3 = [w for p in processed for w in p["final"]]
S4 = [porter.stem(w) for w in S3]

stats = []
for name, words in [("S1 All Tokens", S1), ("S2 No Punctuation", S2),
                    ("S3 Final Tokens", S3), ("S4 Porter Stems", S4)]:
    total = len(words)
    unique = len(set(words))
    ttr = unique / total if total else 0
    stats.append({"Corpus": name, "Total Tokens": total,
                  "Unique Tokens": unique, "TTR": round(ttr, 4)})

display(pd.DataFrame(stats))
print("Top 10 words in S1:", Counter(S1).most_common(10))
print("Top 10 words in S3:", Counter(S3).most_common(10))
print("Top 5 bigrams in S1:", Counter(bigrams(S1)).most_common(5))
print("Top 5 bigrams in S3:", Counter(bigrams(S3)).most_common(5))

frequencies = sorted(Counter(S3).values(), reverse=True)
ranks = np.arange(1, len(frequencies) + 1)
plt.figure(figsize=(7, 4))
plt.plot(np.log(ranks), np.log(frequencies), marker='o', markersize=3)
plt.xlabel("log(rank)")
plt.ylabel("log(frequency)")
plt.title("Zipf plot for S3")
plt.grid(True)
plt.show()

print("TTR of S1:", stats[0]["TTR"])
print("TTR of S4:", stats[3]["TTR"])
print("Removing punctuation and stopwords changes the total number of tokens")
print("Porter stemming can merge multiple forms of a word into one stem")

**Observe:** TTR is unique tokens divided by total tokens. It changes because punctuation and stopwords are removed, and stemming can combine different forms into the same word. The exact values for this dataset are shown in the table.

---
## Q6. Interpretation
Short answers based on the generated tickets

In [ ]:
print("a) TTR changes because we remove punctuation and stopwords, then stem some words into the same form.")
print("b) Example: isn't. My tokenizer keeps it together while word_tokenize splits it.")
print("c) Over stemming example from the dataset:")
examples = compare_df[compare_df["Lancaster"] != compare_df["WordNet"]]
if not examples.empty:
    r = examples.iloc[0]
    print(r["Word"], "becomes", r["Lancaster"], "with Lancaster")
print("d) POS-aware lemmatization keeps a proper word, unlike some shortened stems.")
for word in ["running", "working", "crashes", "better"]:
    if word in unique_words:
        tag = pos_tag([word])[0][1]
        print("Example:", word, "Porter:", porter.stem(word),
              "WordNet:", lemma.lemmatize(word, get_wordnet_pos(tag)))
        break